# ClearerVoice-Studio 语音降噪 - Pitt数据集

## 1. 导入库和环境检查

In [1]:
import warnings
from pathlib import Path
from tqdm import tqdm
import os
import gc
import time
import numpy as np
import soundfile as sf
from scipy import signal
import torch

# 过滤警告
warnings.filterwarnings('ignore')

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
print(f"MPS可用: {hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()}")

PyTorch版本: 2.8.0
CUDA可用: False
MPS可用: True


## 2. 配置路径

In [ ]:
# 输入和输出目录
input_dir = Path('data/raw/Pitt')
output_dir = Path('data/processed/Pitt-FRCRN_SE')

# 获取文件列表
control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

print(f"Control组文件数: {len(control_files)}")
print(f"Dementia组文件数: {len(dementia_files)}")

Control组文件数: 7
Dementia组文件数: 29


## 3. 加载 ClearVoice 模型

In [3]:
from clearvoice import ClearVoice

# 初始化 ClearVoice 模型
# 可选模型:
# - 'FRCRN_SE_16K': 快速，16kHz (推荐)
# - 'MossFormerGAN_SE_16K': 高质量，16kHz
# - 'MossFormer2_SE_48K': 高保真，48kHz

model_name = 'FRCRN_SE_16K'
target_sr = 16000  # 目标采样率，必须与模型匹配

print(f"加载模型: {model_name}...")
if os.path.exists("checkpoints/FRCRN_SE_16K"):
    myClearVoice = ClearVoice(
        task='speech_enhancement',
        model_names=[model_name]
    )
print(f"✓ 模型加载完成")

加载模型: FRCRN_SE_16K...
✓ 模型加载完成


## 4. 显存管理辅助函数

In [4]:
def clear_memory():
    """
    清理显存和内存
    支持 CUDA 和 MPS 后端
    """
    # 清理 Python 垃圾回收
    gc.collect()
    
    # 清理 PyTorch 缓存
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()
        torch.mps.synchronize()
    
    # 短暂等待，确保清理完成
    time.sleep(0.1)


def get_memory_info():
    """
    获取显存使用信息（仅用于调试）
    """
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3  # GB
        reserved = torch.cuda.memory_reserved() / 1024**3
        return f"CUDA - 已分配: {allocated:.2f} GB, 已保留: {reserved:.2f} GB"
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        allocated = torch.mps.current_allocated_memory() / 1024**3
        return f"MPS - 已分配: {allocated:.2f} GB"
    return "CPU - 无显存统计"

## 5. 定义降噪函数

In [5]:
def denoise_audio(audio_path, model, target_sr=16000):
    """
    使用 ClearerVoice 进行语音降噪和增强
    
    Args:
        audio_path: 输入音频文件路径
        model: ClearVoice 模型实例
        target_sr: 目标采样率（16000 或 48000，取决于模型）
    
    Returns:
        denoised_audio: 降噪后的音频 numpy array
        sr: 采样率
    """
    # 加载音频
    audio, sr = sf.read(str(audio_path))
    
    # 如果是多声道，先转为单声道（在重采样之前）
    if len(audio.shape) == 2:
        # audio 形状是 (samples, channels)
        audio = np.mean(audio, axis=1)
    
    # 重采样到目标采样率（使用 scipy，更稳定）
    if sr != target_sr:
        # 计算目标样本数
        num_samples = int(len(audio) * target_sr / sr)
        audio = signal.resample(audio, num_samples)
    
    # 确保是 float32 类型
    audio = audio.astype(np.float32)
    
    # 转换为 [batch, length] 格式
    audio = np.reshape(audio, [1, audio.shape[0]])
    
    # 应用 ClearVoice 降噪
    # 使用 torch.no_grad() 禁用梯度计算，节省显存
    with torch.no_grad():
        # online_write=False 表示返回 numpy 数组而不是直接写入文件
        output_wav = model(audio, online_write=False)
    
    # output_wav 形状: [batch, length]
    return output_wav[0, :], target_sr

## 6. 定义批量处理函数（带显存清理）

In [6]:
def batch_denoise(files, output_subdir, model, target_sr, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）
    
    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        model: ClearVoice 模型实例
        target_sr: 目标采样率
        group_name: 组名（用于显示进度）
    """
    # 创建输出目录
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    success_count = 0
    skip_count = 0
    fail_count = 0
    
    for audio_file in tqdm(files, desc=f"降噪处理 {group_name}"):
        output_file = output_subdir / audio_file.name
        
        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue
        
        try:
            # ⚡ 处理前清理显存
            clear_memory()
            
            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, model, target_sr)
            
            # 保存（16位整数格式）
            sf.write(str(output_file), denoised_audio, sr, subtype='PCM_16')
            success_count += 1
            
            # ⚡ 处理后立即清理显存
            del denoised_audio  # 删除大数组
            clear_memory()
            
        except Exception as e:
            fail_count += 1
            print(f"\n✗ 处理失败: {audio_file.name}: {e}")
            # ⚡ 失败后也要清理显存
            clear_memory()
    
    # 打印统计信息
    print(f"\n{group_name} 处理完成:")
    print(f"  ✓ 成功: {success_count}")
    print(f"  ⊘ 跳过: {skip_count}")
    print(f"  ✗ 失败: {fail_count}")
    print(f"  Σ 总计: {len(files)}")

## 7. 执行批量降噪处理

In [7]:
# 记录开始时间
start_time = time.time()

# ⚡ 开始前先清理显存
print("初始显存状态:", get_memory_info())
clear_memory()
print("清理后显存:", get_memory_info())

# 处理 Dementia 组
print("\n" + "="*60)
print("开始处理 Dementia 组")
print("="*60)
batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    myClearVoice,
    target_sr=target_sr,
    group_name='Dementia'
)

# ⚡ 两组之间清理显存
clear_memory()
print("\n组间清理后显存:", get_memory_info())

# 处理 Control 组
print("\n" + "="*60)
print("开始处理 Control 组")
print("="*60)
batch_denoise(
    control_files,
    output_dir / 'Control',
    myClearVoice,
    target_sr=target_sr,
    group_name='Control'
)

# 计算总时间
elapsed_time = time.time() - start_time
print("\n" + "="*60)
print(f"✓ 所有处理完成！")
print(f"总耗时: {elapsed_time/60:.2f} 分钟 ({elapsed_time:.2f} 秒)")
print(f"输出目录: {output_dir}")
print(f"最终显存状态: {get_memory_info()}")
print("="*60)

初始显存状态: MPS - 已分配: 0.05 GB
清理后显存: MPS - 已分配: 0.05 GB

开始处理 Dementia 组


降噪处理 Dementia:   3%|▎         | 1/29 [00:00<00:25,  1.10it/s]


✗ 处理失败: 125-0.wav: Cannot interpret '2335608' as a data type


降噪处理 Dementia:   7%|▋         | 2/29 [00:01<00:26,  1.02it/s]


✗ 处理失败: 247-0.wav: Cannot interpret '2551256' as a data type


...


✗ 处理失败: 209-3.wav: Cannot interpret '1966920' as a data type


降噪处理 Control:  86%|████████▌ | 6/7 [00:05<00:00,  1.08it/s]


✗ 处理失败: 128-3.wav: Cannot interpret '2719656' as a data type


降噪处理 Control: 100%|██████████| 7/7 [00:06<00:00,  1.05it/s]


✗ 处理失败: 128-2.wav: Cannot interpret '2122308' as a data type

Control 处理完成:
  ✓ 成功: 0
  ⊘ 跳过: 0
  ✗ 失败: 7
  Σ 总计: 7

✓ 所有处理完成！
总耗时: 0.57 分钟 (34.33 秒)
输出目录: data/processed/Pitt-fix-clearvoice
最终显存状态: MPS - 已分配: 0.05 GB
